# Swiss Diptera ID Workbench v0.3 — Colab trainer

Заполни только первую code-cell и выбери **Runtime → Run all**. По умолчанию notebook запускает маленький реальный iNaturalist proof-of-training. Foundation mode использует уже подготовленные official bulk tables из Google Drive и не делает скрытый многотерабайтный download.

In [ ]:
REPO_URL = 'https://github.com/YOUR_NAME/swiss-diptera-id-workbench.git'
MODE = 'quick'  # 'quick' или 'foundation'
INAT_PAGES = 3
INAT_PER_PAGE = 100
FOUNDATION_CONFIG = 'configs/pilot.json'


In [ ]:
from pathlib import Path
import os, subprocess, sys

if 'YOUR_NAME' in REPO_URL:
    raise ValueError('В первой cell замени YOUR_NAME на свой GitHub username')
project = Path('/content/swiss-diptera-id-workbench')
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(project)], check=True)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-corpus.txt'], check=True)
print('Project ready:', project)


In [ ]:
if MODE == 'quick':
    subprocess.run([sys.executable, 'scripts/build_inat_pilot.py', '--pages', str(INAT_PAGES), '--per-page', str(INAT_PER_PAGE), '--download'], check=True)
    subprocess.run([sys.executable, 'scripts/embed_dataset.py', '--manifest', 'data/inat_pilot_manifest.csv', '--out-dir', 'models'], check=True)
    subprocess.run([sys.executable, 'scripts/train_multidomain.py', '--model-dir', 'models', '--min-images-per-class', '2', '--out', 'models/classifiers.joblib'], check=True)
    subprocess.run([sys.executable, 'scripts/build_retrieval_index.py', '--model-dir', 'models'], check=True)
    print('Quick proof complete. Models are in /content/swiss-diptera-id-workbench/models')


## Foundation mode

Сначала положи official source files в Drive и укажи эти пути в `configs/pilot.json`. BIOSCAN/GBIF/DiSSCo требуют своих download/export steps, описанных в `ONE_COMMAND_TRAINING_RU.md`.

In [ ]:
if MODE == 'foundation':
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, 'scripts/prepare_corpus.py', '--config', FOUNDATION_CONFIG], check=True)
    subprocess.run([sys.executable, 'scripts/download_images.py', '--manifest', 'data/corpus/pilot_manifest.csv', '--out', 'data/corpus/pilot_downloaded.csv', '--image-root', 'data/images/pilot'], check=True)
    subprocess.run([sys.executable, 'scripts/embed_dataset.py', '--manifest', 'data/corpus/pilot_downloaded.csv', '--out-dir', 'models'], check=True)
    subprocess.run([sys.executable, 'scripts/train_multidomain.py', '--model-dir', 'models'], check=True)
    subprocess.run([sys.executable, 'scripts/build_retrieval_index.py', '--model-dir', 'models'], check=True)
    print('Foundation pilot complete.')
